In [1]:
from pathlib import Path
from typing import Callable

import numpy as np
import prism
import xarray as xr

from imagematerials.stock import compute_dynamic_stock_driven, compute_historic
from imagematerials.survival import ScipySurvival, SurvivalMatrix, convert_life_time_vehicles
from imagematerials.vehicles.preprocessing import preprocessing


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

In [3]:


def export_to_netcdf(prep_data, out_fp):
    simple_datasets = {key: val for key, val in prep_data.items() if key.endswith("_simple")}
    typical_datasets = {key: val for key, val in prep_data.items() if key.endswith("_typical")}
    lf_vehicles = convert_life_time_vehicles(prep_data["lifetimes_vehicles"])
    other_datasets = {key: val for key, val in prep_data.items()
                      if key not in ["lifetimes_vehicles"] + list(simple_datasets) + list(typical_datasets)}
    xr.Dataset(simple_datasets).to_netcdf(out_fp, group="simple", engine="netcdf4")
    xr.Dataset(typical_datasets).to_netcdf(out_fp, group="typical", mode="a", engine="netcdf4")
    xr.Dataset(lf_vehicles).to_netcdf(out_fp, group="lifetimes", mode="a", engine="netcdf4")
    xr.Dataset(other_datasets).to_netcdf(out_fp, group="other", mode="a", engine="netcdf4")

def import_from_netcdf(in_fp):
    lt = xr.open_dataset(in_fp, group="lifetimes", engine="netcdf4").load()
    return {
        "simple": xr.open_dataset(in_fp, group="simple", engine="netcdf4").load(),
        "typical": xr.open_dataset(in_fp, group="typical", engine="netcdf4").load(),
        "lifetimes": {dist_name: arr.dropna("mode") for dist_name, arr in lt.items()},
        "other": xr.open_dataset(in_fp, group="other", engine="netcdf4").load(),
    }

In [4]:
if not prep_fp.is_file():
    _, prep_data = preprocessing(base_dir)
    export_to_netcdf(prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)

/home/qubix/Documents/repos/image-mat/image-materials/imagematerials/vehicles/preprocessing.py:325: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/imagematerials/vehicles/preprocessing.py:325: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/imagematerials/vehicles/preprocessing.py:325: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/imagematerials/vehicles/preprocessing.py:325: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/imagematerials/vehicles/preprocessing.py:325: PerformanceWarning: indexing pas

In [5]:
vehicle_nr = prep_data["simple"]['total_nr_vehicles_simple']
life_time_vehicles = prep_data["lifetimes"]

In [7]:
survival_matrix = SurvivalMatrix(ScipySurvival(life_time_vehicles))

In [8]:
start_simulation = 1970
end_simulation = vehicle_nr.coords["time"].max()
Region = prism.NewDim("region", coords=[str(x) for x in np.arange(1, 29)])
Mode = prism.NewDim("mode", coords=[str(x) for x in vehicle_nr["mode"].to_numpy()])
Cohort = prism.NewDim("cohort", coords=[str(x) for x in vehicle_nr["time"].to_numpy()])

In [9]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    stock: prism.TimeVariable[Region, Mode] #TODO check how to have property that can be both input and output within prism
    stock_function: Callable    # defines the stock function to use e.g. stock or inflow driven

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.stock, self.survival_matrix, self.start_simulation,
                         self.stock_by_cohort, self.inflow, self.outflow_by_cohort,
                         self.stock_function)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        self.stock_function(self.stock, self.stock_by_cohort,  self.inflow, self.outflow_by_cohort, self.survival_matrix, t)

In [10]:
timeline = prism.Timeline(start=vehicle_nr.coords["time"][0],
                          end=end_simulation, stepsize=1)
timeline_simulate = prism.Timeline(start=start_simulation,
                          end=end_simulation, stepsize=1)

In [11]:
model = Stocks(timeline, start_simulation=start_simulation, survival_matrix=survival_matrix, stock=vehicle_nr, stock_function = compute_dynamic_stock_driven)

In [12]:
model.simulate(timeline_simulate)